# Doğrusal Regresyon

Bu notebook, doğrusal regresyon algoritmasının hem tek değişkenli hem de çok değişkenli versiyonlarını uygulamalı olarak göstermektedir. Gradient descent (dereceli alçalma) optimizasyon tekniği kullanılarak model parametreleri adım adım optimize edilmektedir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Veri Seti Oluşturma ve Görselleştirme

Aşağıda, doğrusal regresyon modelimiz için bir sentetik veri seti oluşturuyoruz. Gerçek hayatta karşılaşacağımız verileri taklit etmek için verilere rastgele gürültü ekliyoruz.

Kullandığımız model: $y = \theta_0 + \theta_1 x + \epsilon$

Burada:
- $\theta_0 = -3$ (kesişim/bias/intercept)
- $\theta_1 = 0.5$ (eğim/weights/slope)
- $\epsilon \sim \mathcal{N}(0, 0.05)$ (standart sapması 0.05 olan normal dağılımlı gürültü)

In [ ]:
ornek_sayisi = 100
theta_0 = -3
theta_1 = 0.5
X = np.linspace(-2, 2, ornek_sayisi)
y = theta_0 + (theta_1 * X) + np.random.normal(0, 0.05, size=ornek_sayisi)
print(X.shape, y.shape)

Oluşturduğumuz verileri görselleştirelim. Bu grafik, X ve y değişkenleri arasındaki doğrusal ilişkiyi gösterir.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X, y)
plt.xlabel("X", fontsize=18)
plt.ylabel("y", fontsize=18)
plt.show()

## Eğitim ve Doğrulama Veri Setlerinin Ayrılması

Model performansını doğru bir şekilde değerlendirmek için, veri setimizi eğitim (%80) ve doğrulama (%20) olarak ikiye ayırıyoruz. Eğitim verisiyle modeli eğitecek, doğrulama verisiyle de modelin genelleme yeteneğini test edeceğiz.

In [ ]:
# verileri %80 - %20 olacak şekilde eğitim ve doğrulama verisi olarak ayır
egitim_ornek_sayisi = int(ornek_sayisi * 0.8)

X_egitim, X_val = X[:egitim_ornek_sayisi], X[egitim_ornek_sayisi:]
y_egitim, y_val = y[:egitim_ornek_sayisi], y[egitim_ornek_sayisi:]

print(f"X egitim: {X_egitim.shape}, X validasyon: {X_val.shape}")
print(f"y egitim: {y_egitim.shape}, y validasyon: {y_val.shape}")

## Doğrusal Regresyon ve Dereceli Alçalma

Doğrusal regresyonda, verilen özellik (feature) değerlerine dayalı olarak hedef değişkenin değerini tahmin etmeye çalışırız. Tek değişkenli doğrusal regresyon için matematiksel model, formül ve optimizasyon sürecini aşağıda görebilirsiniz.

$h_{\theta}(x) = \theta_0 + \theta_1x$<br><br>
$J(\theta) = \frac{1}{2m} \sum\limits_{i=1}^m (h_{\theta}(x^{(i)}) - y^{(i)})^2$<br><br>
$\frac{\partial}{\partial \theta_0}J(\theta) = \frac{1}{m} \sum\limits_{i=1}^m (\theta_0+\theta_1x^{(i)}-y^{(i)})$<br><br>
$\frac{\partial}{\partial \theta_1}J(\theta) = \frac{1}{m} \sum\limits_{i=1}^m (\theta_0+\theta_1x^{(i)}-y^{(i)}) \cdot x^{(i)}$<br>
<br><br>
$\theta_j := \theta_j - \alpha \frac{\partial}{\partial \theta_j}J(\theta)$


## Tek Özellik (Feature) için Doğrusal Regresyon Sınıfı

Aşağıda, tek bir özellik için doğrusal regresyon modeli oluşturan bir sınıf tanımlıyoruz. Bu sınıf:
- Gradient descent algoritmasını kullanarak $\theta_0$ ve $\theta_1$ parametrelerini optimize eder
- Eğitim sürecinde modelin gelişimini görselleştirir (her 50 iterasyonda bir)

In [ ]:
# https://github.com/python-engineer/MLfromscratch/ adresindeki örnekten uyarlanmıştır.


def hata_fonksiyonu(y_gercek, y_tahmin):
    ornek_sayisi = len(y_gercek)
    hata = 1 / (2 * ornek_sayisi) * np.sum((y_tahmin - y_gercek) ** 2)
    return hata


class LinearRegression_1_feature:
    def __init__(self, ogrenme_hizi):
        self.ogrenme_hizi = ogrenme_hizi
        self.theta_0 = None
        self.theta_1 = None

    def fit(self, X, y, tekrar_sayisi):
        ornek_sayisi = len(X)

        # parametrelere ilk değerlerini ata
        self.theta_0 = 0
        self.theta_1 = 0

        # grafiği hazırla
        plt.figure(figsize=(8, 8))
        plt.scatter(X, y)
        plt.xlabel("X", fontsize=18)
        plt.ylabel("y", fontsize=18)

        hatalar = []

        # gradient descent
        for i in range(tekrar_sayisi):
            y_tahmin = self.theta_0 + (self.theta_1 * X)

            # türevleri hesapla
            d_theta_0 = (1 / ornek_sayisi) * np.sum(y_tahmin - y)
            d_theta_1 = (1 / ornek_sayisi) * np.dot(X.T, (y_tahmin - y))

            # parametreleri güncelle
            self.theta_0 = self.theta_0 - self.ogrenme_hizi * d_theta_0
            self.theta_1 = self.theta_1 - self.ogrenme_hizi * d_theta_1

            # her 50 döngüde bir tahmin grafiğini göster
            if i % 50 == 0:
                plt.plot(X, y_tahmin)
                hata = hata_fonksiyonu(y, y_tahmin)
                hatalar.append(hata)

        plt.show()
        return hatalar

    def predict(self, X):
        y_tahmin = self.theta_0 + (self.theta_1 * X)
        return y_tahmin

    def get_params(self):
        return self.theta_0, self.theta_1


## Model Oluşturma ve Eğitme

Şimdi, tanımladığımız sınıfı kullanarak bir model oluşturacak ve eğitim verileri üzerinde eğiteceğiz.

In [ ]:
model = LinearRegression_1_feature(0.01)

Modeli eğitim veri seti üzerinde 1000 iterasyon boyunca eğitiyoruz. Her 50 iterasyonda bir modelin tahminleri grafikte gösterilir ve hata değeri kaydedilir.

In [ ]:
hatalar = model.fit(X_egitim, y_egitim, 1000)

## Hata Eğrisi

Eğitim sırasında kaydedilen hata değerlerini zaman içinde grafiğe çizelim. Bu grafik, modelimizin eğitim sırasında nasıl geliştiğini gösterir.

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(range(len(hatalar)), hatalar)
plt.xlabel("Zaman", fontsize=18)
plt.ylabel(r"$J(\theta)$", fontsize=18)
plt.show()

## Eğitilmiş Model

Eğitilen modelimizin, eğitim veri seti üzerindeki performansını görselleştirelim. Kırmızı çizgi, modelimizin tahminlerini gösterir.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X_egitim, y_egitim)

plt.xlabel("X", fontsize=18)
plt.ylabel("y", fontsize=18)

y_tahmin = model.predict(X_egitim)
plt.plot(X_egitim, y_tahmin, color="red", linewidth=4)

plt.show()

## Model Doğrulama (Validasyon)

Şimdi, eğitilmiş modelimizi daha önce görmediği doğrulama verisi üzerinde test edelim. Bu, modelimizin genelleme yeteneğini değerlendirmemize yardımcı olur.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X_val, y_val)

plt.xlabel("X", fontsize=18)
plt.ylabel("y", fontsize=18)

plt.plot(X_val, model.predict(X_val), color="red", linewidth=4)

plt.show()

## Model Performansının Değerlendirilmesi

Model performansını değerlendirmek için Ortalama Kare Hata (Mean Squared Error - MSE) metriğini kullanacağız.

In [ ]:
def mean_squared_error(y_gercek, y_tahmin):
    return np.mean((y_gercek - y_tahmin) ** 2)

In [ ]:
print(f"MSE: {mean_squared_error(y_val, model.predict(X_val))}")

Modelimizin öğrendiği parametreleri alalım. Bunların, veri setini oluştururken kullandığımız gerçek değerlere ($\theta_0 = -3$ ve $\theta_1 = 0.5$) ne kadar yakın olduğunu görebiliriz.

In [ ]:
print(f"Modelin öğrendiği parametreler: {model.get_params()}")

# Çok Değişkenli Doğrusal Regresyon

Şimdi, birden fazla özelliğe (feature) sahip veriler için doğrusal regresyon modelini uygulayacağız. 

## Veri Seti Hazırlama

Çok değişkenli doğrusal regresyon için sentetik bir veri seti oluşturacağız. 3 boyutlu bir özellik vektörü (ilki bias terimi için 1) ve 3 boyutlu bir parametre vektörü ($\theta_0, \theta_1, \theta_2$) kullanacağız.

In [ ]:
ornek_sayisi = 100
theta_arr = np.array([3, -2, 5])

X = np.ones((ornek_sayisi, 3))
X[:,1] = np.linspace(-2, 1, ornek_sayisi).T
X[:,2] = np.linspace(1, 2, ornek_sayisi).T

y = np.dot(X, theta_arr.T)

print("İlk 5 X değeri:")
print(X[:5])
print("İlk 5 y değeri:")
print(y[:5])

Verilerimizin boyutlarını kontrol edelim. X matrisimiz (100, 3) boyutunda olmalı (100 örnek, her biri 3 özellik içerir) ve y vektörümüz (100,) boyutunda olmalıdır.

In [ ]:
print(X.shape, y.shape)

## Çok Değişkenli Doğrusal Regresyon Sınıfı

Şimdi, çok değişkenli doğrusal regresyon için genel bir sınıf tanımlayacağız. Bu sınıf, daha önceki tek değişkenli sınıfın genelleştirilmiş halidir ve herhangi bir boyuttaki özellik vektörüyle çalışabilir.

In [ ]:
class LinearRegression:
    def __init__(self, ogrenme_hizi):
        self.ogrenme_hizi = ogrenme_hizi
        self.thetalar = None

    def fit(self, X, y, tekrar_sayisi):
        ornek_sayisi, feature_sayisi = X.shape

        # parametrelere ilk değerlerini ata
        self.thetalar = np.zeros(feature_sayisi)
        hatalar = []

        # gradient descent
        for i in range(tekrar_sayisi):
            y_tahmin = np.dot(X, self.thetalar.T)

            # türevleri hesapla
            d_thetalar = (1 / ornek_sayisi) * np.dot(X.T, (y_tahmin - y))

            # parametreleri güncelle
            self.thetalar = self.thetalar - self.ogrenme_hizi * d_thetalar

            # her 20 döngüde bir hatayı kaydet
            if i % 20 == 0:
                hata = hata_fonksiyonu(y, y_tahmin)
                hatalar.append(hata)

        return hatalar

    def predict(self, X):
        y_tahmin = np.dot(X, self.thetalar.T)
        return y_tahmin

    def get_params(self):
        return self.thetalar

## Çok Değişkenli Model Eğitimi

Çok değişkenli doğrusal regresyon modelimizi oluşturuyoruz ve eğitim verisi üzerinde eğitiyoruz.

In [ ]:
lr_model = LinearRegression(0.01)
hatalar = lr_model.fit(X, y, 500)

## Hata Eğrisi (Çok Değişkenli Model)

Çok değişkenli modelimizin eğitim sırasındaki hata değerlerinin değişimini grafiğe çizelim.

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(range(len(hatalar)), hatalar)
plt.xlabel("Zaman", fontsize=18)
plt.ylabel(r"$J(\theta)$", fontsize=18)
plt.show()

## Öğrenilen Parametreler (Çok Değişkenli Model)

Çok değişkenli modelimizin öğrendiği parametreleri görüntüleyelim ve bunları gerçek değerlerle ([3, -2, 5]) karşılaştıralım.

In [ ]:
print("Öğrenilen Theta değerleri (doğru değerler: [3, -2, 5]:")
print(lr_model.get_params())